## Book Sorting based on Category

Here we will create a 'book sorter' based on specified category. The process includes:

1.  **Defining Fiction Category**: A list of categories you want to sort books by.
2.  **Preparing Book Data**: A simulation of book titles and short descriptions.
3.  **Embedding Category and Book Descriptions**: Use `SentenceTransformer` to convert both category and book descriptions into numerical vectors.
4.  **Calculating Cosine Similarity**: Measure the similarity between each book description and each category.
5.  **Assigning Category**: Assign the category with the highest similarity to each book.

### **Code Description: Data Initialization**
This cell initializes the data required for the book sorting application. It performs the following actions:
- **Imports `pandas`**: The `pandas` library is imported for efficient data manipulation, particularly for creating and managing the `fiction_books` DataFrame.
- **Defines `book_category`**: A Python list named `book_category` is created. This list contains a comprehensive set of categories and genres that will be used to classify books.
- **Prepares `fiction_books` DataFrame**: A `pandas` DataFrame called `fiction_books` is constructed. This DataFrame holds sample book data, including titles, authors, and brief descriptions, which are essential for the category assignment process.

### **Code Description: Data Initialization**
This cell initializes the data required for the book sorting application. It performs the following actions:
- **Imports `pandas`**: The `pandas` library is imported for efficient data manipulation, particularly for creating and managing the `fiction_books` DataFrame.
- **Defines `book_category`**: A Python list named `book_category` is created. This list contains a comprehensive set of categories and genres that will be used to classify books.
- **Prepares `fiction_books` DataFrame**: A `pandas` DataFrame called `fiction_books` is constructed. This DataFrame holds sample book data, including titles, authors, and brief descriptions, which are essential for the category assignment process.

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

projectFiles = '/content/drive/My Drive/HCAI1/BooksDataset/'
# you can get this by going to files after you have mounted the drive,
# right clicking the folder you want to use, then selecting "copy path"

Mounted at /content/drive/


In [2]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Defining fiction category
book_category = [
    "fantasy", "sci-fi", "romance", "horror", "thriller",
    "young adult", "drama", "mystery", "adventure",
    "historical fiction", "biography", "poetry", "young adult fiction"
]

# The fiction_books DataFrame will be loaded from the CSV file in a later cell.
# The example data definition is removed as requested.

print("Book category are defined. Book data will be loaded from CSV.")

Book category are defined. Book data will be loaded from CSV.


### **Loading Book Data from CSV File**

This cell loads `BooksDataset.csv` from your Google Drive and replaces the existing `fiction_books` DataFrame. It's important to ensure that the CSV file contains relevant columns such as 'Title', 'Author', and 'Description', which will be renamed to match the expected data structure.

After running this cell, you must **re-run** the subsequent cells that handle book embedding and Gradio interface setup for the new data to be reflected.

In [3]:
csv_file_path = f'{projectFiles}BooksDataset.csv'

# Load the CSV file
try:
    print("debug 1", csv_file_path)
    new_books_df = pd.read_csv(csv_file_path)
    print(f"'BooksDataset.csv' loaded successfully from: {csv_file_path}")
    print("First 5 rows of the new DataFrame:")
    display(new_books_df.head())
    print("Information about the new DataFrame:")
    new_books_df.info()
except FileNotFoundError:
    print(f"Error: 'BooksDataset.csv' was not found at path: {csv_file_path}")
    print("Please ensure the file is correctly placed in your Google Drive under 'HCAI1/'.")
except Exception as e:
    print(f"An error occurred while loading the CSV file: {e}")

debug 1 /content/drive/My Drive/HCAI1/BooksDataset/BooksDataset.csv
'BooksDataset.csv' loaded successfully from: /content/drive/My Drive/HCAI1/BooksDataset/BooksDataset.csv
First 5 rows of the new DataFrame:


,Title,Authors,Description,Category,Publisher,Publish Date,Price
0,Goat Brothers,"By Colton, Larry",NaN,"History , General",Doubleday,"Friday, January 1, 1993",Price Starting at $8.79
1,The Missing Person,"By Grumbach, Doris",NaN,"Fiction , General",Putnam Pub Group,"Sunday, March 1, 1981",Price Starting at $4.99
2,Don't Eat Your Heart Out Cookbook,"By Piscatella, Joseph C.",NaN,"Cooking , Reference",Workman Pub Co,"Thursday, September 1, 1983",Price Starting at $4.99
3,When Your Corporate Umbrella Begins to Leak: A...,"By Davis, Paul D.",NaN,NaN,Natl Pr Books,"Monday, April 1, 1991",Price Starting at $4.99
4,Amy Spangler's Breastfeeding : A Parent's Guide,"By Spangler, Amy",NaN,NaN,Amy Spangler,"Saturday, February 1, 1997",Price Starting at $5.32


Information about the new DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103082 entries, 0 to 103081
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   Title         103082 non-null  object
 1   Authors       103082 non-null  object
 2   Description   70213 non-null   object
 3   Category      76912 non-null   object
 4   Publisher     103074 non-null  object
 5   Publish Date  103082 non-null  object
 6   Price         103082 non-null  object
dtypes: object(7)
memory usage: 5.5+ MB


In [4]:
# Replace the existing fiction_books DataFrame with the new data
# Make sure to adjust column names to match your CSV file.
# I'm assuming your CSV has 'Title', 'Authors', 'Description' columns.
# If your column names are different, please modify them below.

if 'new_books_df' in locals() and not new_books_df.empty:
    # Rename columns to match the expected structure
    # Adjust these names if your CSV has different column names
    column_mapping = {
        'Title': 'title',
        'Authors': 'author', # Corrected from 'Author' to 'Authors' based on CSV info
        'Description': 'description'
    }

    # Filter only columns that exist in the new DataFrame to avoid errors
    columns_to_rename = {k: v for k, v in column_mapping.items() if k in new_books_df.columns}

    # Check if all *expected* columns (title, author, description) from the mapping are present
    # in the new_books_df after applying the mapping.
    # This ensures that even if the original CSV had more columns, we only proceed if the crucial ones are there.
    required_mapped_columns = ['title', 'author', 'description']
    if all(col in new_books_df.rename(columns=columns_to_rename).columns for col in required_mapped_columns):
        fiction_books = new_books_df.rename(columns=columns_to_rename)
        fiction_books = fiction_books[required_mapped_columns] # Ensure correct order and only necessary columns

        # Fill NaN values in the 'description' column with an empty string
        fiction_books['description'] = fiction_books['description'].fillna('')

        # Add an empty 'assigned_category' column, which will be populated later
        fiction_books['assigned_category'] = [[]] * len(fiction_books)

        print("\n'fiction_books' DataFrame has been updated with data from the CSV file.")
        display(fiction_books.head())
        print("\n*** IMPORTANT: Now, please re-run the book embedding cell (Embedding Category and Book Descriptions) and then the Gradio interface again to update the system with the new data. ***")
    else:
        print("Error: 'BooksDataset.csv' does not contain the expected columns ('Title', 'Authors', 'Description') or has different names after mapping.")
        print("Please check your CSV file and adjust the 'column_mapping' in the code.")
else:
    print("No data to process as 'new_books_df' was not loaded correctly or is empty.")


'fiction_books' DataFrame has been updated with data from the CSV file.


,title,author,description,assigned_category
0,Goat Brothers,"By Colton, Larry",,[]
1,The Missing Person,"By Grumbach, Doris",,[]
2,Don't Eat Your Heart Out Cookbook,"By Piscatella, Joseph C.",,[]
3,When Your Corporate Umbrella Begins to Leak: A...,"By Davis, Paul D.",,[]
4,Amy Spangler's Breastfeeding : A Parent's Guide,"By Spangler, Amy",,[]



*** IMPORTANT: Now, please re-run the book embedding cell (Embedding Category and Book Descriptions) and then the Gradio interface again to update the system with the new data. ***


### **Code Description: Install Gradio**
This cell installs the `gradio` library, which is used to create interactive web interfaces for machine learning models. The `-q` flag ensures a quiet installation, suppressing most output messages.

### **Code Description: Install Gradio**
This cell installs the `gradio` library, which is used to create interactive web interfaces for machine learning models. The `-q` flag ensures a quiet installation, suppressing most output messages.

In [5]:
pip install -q gradio

### **Code Description: Embedding and Similarity Calculation**
This cell is central to the category assignment process. It performs the following key steps:
- **Imports Libraries**: It imports `SentenceTransformer` for converting text into numerical embeddings and `cosine_similarity` from `sklearn.metrics.pairwise` to measure the similarity between these embeddings.
- **Loads Pre-trained Model**: It loads the `all-MiniLM-L6-v2` pre-trained Sentence Transformer model, which is highly efficient for generating high-quality sentence embeddings.
- **Embeds Category and Book Descriptions**: Both the `book_category` and the `fiction_books['description']` are converted into numerical vector representations using the loaded `SentenceTransformer` model. These embeddings capture the semantic meaning of the text.
- **Calculates Cosine Similarity**: A `similarity_matrix` is computed using `cosine_similarity`. This matrix quantifies how similar each book's description is to each predefined category.
- **Assigns Top Category**: The `assign_top_category` function is defined and applied to each book. This function identifies the top N (defaulting to 2) most similar category for each book based on the `similarity_matrix` and stores them in a new column called `assigned_category` within the `fiction_books` DataFrame. The results are then printed, showing each book with its author and assigned category along with their similarity scores.

### **Code Description: Embedding and Similarity Calculation**
This cell is central to the category assignment process. It performs the following key steps:
- **Imports Libraries**: It imports `SentenceTransformer` for converting text into numerical embeddings and `cosine_similarity` from `sklearn.metrics.pairwise` to measure the similarity between these embeddings.
- **Loads Pre-trained Model**: It loads the `all-MiniLM-L6-v2` pre-trained Sentence Transformer model, which is highly efficient for generating high-quality sentence embeddings.
- **Embeds Category and Book Descriptions**: Both the `book_category` and the `fiction_books['description']` are converted into numerical vector representations using the loaded `SentenceTransformer` model. These embeddings capture the semantic meaning of the text.
- **Calculates Cosine Similarity**: A `similarity_matrix` is computed using `cosine_similarity`. This matrix quantifies how similar each book's description is to each predefined category.
- **Assigns Top Category**: The `assign_top_category` function is defined and applied to each book. This function identifies the top N (defaulting to 2) most similar category for each book based on the `similarity_matrix` and stores them in a new column called `assigned_category` within the `fiction_books` DataFrame. The results are then printed, showing each book with its author and assigned category along with their similarity scores.

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Ensure the model is loaded
# If 'model' from previous cells is still available, reuse it.
# Otherwise, load it again.
if 'model' not in locals():
    model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Embedding category and book descriptions
theme_vectors = model.encode(book_category, show_progress_bar=True)
book_description_vectors = model.encode(fiction_books['description'].tolist(), show_progress_bar=True)

print("Category and book descriptions are now embedded.")

# 4. Calculating Cosine Similarity
# Calculates similarity between each book description and each category
similarity_matrix = cosine_similarity(book_description_vectors, theme_vectors)

# 5. Assigning category to books
def assign_top_category(row_similarity_scores, category, num_top_category=2):
    # Get indices of the top N most similar category
    top_indices = row_similarity_scores.argsort()[-num_top_category:][::-1]
    # Return the names of these category and their similarity scores
    return [
        (category[i], row_similarity_scores[i])
        for i in top_indices
    ]

fiction_books['assigned_category'] = [
    assign_top_category(scores, book_category)
    for scores in similarity_matrix
]

print("Top category have been assigned to each book.")

# Display the results
print("\nBook Sorting Results:")
for index, row in fiction_books.iterrows():
    category_str = ', '.join([f"{category_item} ({score:.2f})" for category_item, score in row['assigned_category']])
    print(f"Title: {row['title']}\n  Author: {row['author']}\n  Category: {category_str}\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3222 [00:00<?, ?it/s]

Streaming af output blev afkortet til de sidste 5000 linjer.
Title: Living religion
  Author: By Fisher, Joseph L
  Category: horror (0.38), mystery (0.37)

Title: Young Guns: A New Generation Of Conservative Leaders
  Author: By Cantor, Eric, Ryan, Paul, and McCarthy, Kevin
  Category: young adult fiction (0.23), young adult (0.15)

Title: Great Presidential Decisions: State Papers That Changed the Course of History from Washington to Reagan
  Author: By Morris, Richard Brandon and Morris, Jeffrey B.
  Category: historical fiction (0.30), biography (0.29)

Title: Stealing Heaven
  Author: By Scott, Elizabeth
  Category: young adult fiction (0.25), romance (0.20)

Title: Solar Revolution: The Economic Transformation of the Global Energy Industry (The MIT Press)
  Author: By Bradford, Travis
  Category: sci-fi (0.13), young adult (0.12)

Title: Creatures of the Seashore
  Author: By Kate Londesborough
  Category: horror (0.38), mystery (0.37)

Title: Reasons to Kill: Why Americans Choos

### **Code Description: Install Requests Library**
This cell ensures the `requests` library is installed. This library is used to directly download book texts from Project Gutenberg, providing a more robust method compared to the `gutenberg` library which has shown compatibility issues.

### **Code Description: Retrieve and Integrate Project Gutenberg Book**
This cell demonstrates how to directly download a book from Project Gutenberg using the `requests` library and integrate its details into the `fiction_books` DataFrame.
- **Direct Download**: It fetches the text of a specific book (e.g., 'Pride and Prejudice' with Gutenberg ID 1342) directly from Project Gutenberg's servers.
- **Text Cleaning**: Basic cleaning is performed to remove Project Gutenberg's header and footer information from the raw text.
- **Extract Book Details**: A helper function `get_book_details_from_text` extracts the title, author, and a description (first 500 characters) from the downloaded text.
- **DataFrame Integration**: The extracted book details are formatted into a new DataFrame row and concatenated with the existing `fiction_books` DataFrame, making the new book available for category assignment and search.

### **Code Description: Book Search Function**
This cell defines the `search_books_by_category` function, which is designed to filter books based on selected category. Here's a breakdown of its functionality:
- **Input Parameter**: It takes a list of `selected_category` as input.
- **Input Validation**: It checks if any category are selected; if not, it returns a prompt to the user.
- **Filtering Logic**: It filters the `fiction_books` DataFrame to include only those books that have at least one of their `assigned_category` present in the `selected_category` list.
- **Relevance Scoring and Sorting**: It calculates a `relevance_score` for each filtered book by summing the similarity scores of the selected category. The results are then sorted in descending order of relevance.
- **No Books Found**: If no books match the selected criteria, an appropriate message is returned.
- **Formatted Output**: For matching books, it constructs a formatted string that includes the book's title, author, description, assigned category with their similarity scores, and the calculated relevance score.

### **Code Description: Book Search Function**
This cell defines the `search_books_by_category` function, which is designed to filter books based on selected category. Here's a breakdown of its functionality:
- **Input Parameter**: It takes a list of `selected_category` as input.
- **Input Validation**: It checks if any category are selected; if not, it returns a prompt to the user.
- **Filtering Logic**: It filters the `fiction_books` DataFrame to include only those books that have at least one of their `assigned_category` present in the `selected_category` list.
- **No Books Found**: If no books match the selected criteria, an appropriate message is returned.
- **Formatted Output**: For matching books, it constructs a formatted string that includes the book's title, author, description, and all assigned category with their similarity scores.

In [7]:
import gradio as gr

def search_books_by_category(selected_category):
    if not selected_category:
        return "Please select at least one category."

    # Filter books based on whether any of their assigned category match the selected category
    filtered_books = fiction_books[
        fiction_books['assigned_category'].apply(
            lambda x: any(category_name in selected_category for category_name, _ in x)
        )
    ].copy() # Use .copy() to avoid SettingWithCopyWarning

    if filtered_books.empty:
        return "No books found for the selected category."

    # Calculate a relevance score for each filtered book
    # The relevance score is the sum of similarity scores for the selected category
    filtered_books['relevance_score'] = filtered_books['assigned_category'].apply(
        lambda assigned_list: sum(score for category_name, score in assigned_list if category_name in selected_category)
    )

    # Sort books by relevance score in descending order
    filtered_books = filtered_books.sort_values(by='relevance_score', ascending=False).reset_index(drop=True)

    output_str = ""
    for index, row in filtered_books.iterrows():
        category_str = ', '.join([f"{category_item} ({score:.2f})" for category_item, score in row['assigned_category']])
        output_str += f"Title: {row['title']}\n"
        output_str += f"Author: {row['author']}\n"
        output_str += f"Description: {row['description']}\n"
        output_str += f"Assigned Category: {category_str}\n"
        output_str += f"Relevance Score (sum of selected category similarities): {row['relevance_score']:.2f}\n\n"

    return output_str

### **Code Description: Gradio Interface Setup and Launch**
This cell sets up and launches the interactive web interface for the book sorter using the `gradio` library. Here's how it works:
- **Creates Gradio Interface**: An `iface` object is created using `gr.Interface`.
  - `fn=search_books_by_category`: Specifies that the `search_books_by_category` function (defined in the previous cell) will handle the backend logic when the interface is interacted with.
  - `inputs=gr.CheckboxGroup(choices=book_category, label="Select Category")`: Defines the input component as a group of checkboxes, allowing users to select multiple category from the `book_category` list.
  - `outputs=gr.Textbox(label="Matching Books")`: Defines the output component as a textbox where the results from `search_books_by_category` will be displayed.
  - `title="Book Sorter by Category"` and `description="Select one or more category to see matching books."`: Sets the title and a descriptive message for the Gradio application.
- **Launches Interface**: `iface.launch()` starts the Gradio web server, making the interactive interface accessible, typically via a local URL or a public share link (as indicated by the output in a Colab environment).

### **Code Description: Gradio Interface Setup and Launch**
This cell sets up and launches the interactive web interface for the book sorter using the `gradio` library. Here's how it works:
- **Creates Gradio Interface**: An `iface` object is created using `gr.Interface`.
  - `fn=search_books_by_category`: Specifies that the `search_books_by_category` function (defined in the previous cell) will handle the backend logic when the interface is interacted with.
  - `inputs=gr.CheckboxGroup(choices=book_category, label="Select Category")`: Defines the input component as a group of checkboxes, allowing users to select multiple category from the `book_category` list.
  - `outputs=gr.Textbox(label="Matching Books")`: Defines the output component as a textbox where the results from `search_books_by_category` will be displayed.
  - `title="Book Sorter by Category"` and `description="Select one or more category to see matching books."`: Sets the title and a descriptive message for the Gradio application.
- **Launches Interface**: `iface.launch()` starts the Gradio web server, making the interactive interface accessible, typically via a local URL or a public share link (as indicated by the output in a Colab environment).

In [8]:
iface = gr.Interface(
    fn=search_books_by_category,
    inputs=gr.CheckboxGroup(choices=book_category, label="Select Category"),
    outputs=gr.Textbox(label="Matching Books"),
    title="Book Sorter by Category",
    description="Select one or more category to see matching books."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://72dc9b4dfb409c1dc9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
